# Wireless Coverage — Part 4: Naive Tower Viewsheds → H3 Coverage

With the LiDAR-derived surface tables H3-indexed in Part 3, this notebook answers the
core question: *what can a tower actually see?* Place one candidate tower at the centroid
of each H3 cell (the "naive" lattice), compute the line-of-sight viewshed for each tower
over the DSM surface (buildings and tree canopy block the view), and aggregate back to an
H3 per-cell tower-LOS count — a coverage signal that captures how many towers have a clear
sightline to each location.

> **Runtime.** Runs on **Databricks Serverless environment 6** — the lightweight tier (pure
> Python/PySpark, no JAR or GDAL init script). Set `DEMO=True` (default) for a Golden Gate
> Park sub-AOI (≈25 towers) to iterate quickly; `DEMO=False` for full San Francisco (≈1,150
> towers). Full SF is the shipped deliverable.

---
**Last Update:** 2026-10-03

In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall \
  "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet \
  "geobrix[light_env6,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
%run ./config_nb

In [ ]:
# ── nb4 overrides (after config_nb) ────────────────────────────────────────
# config_nb provides: F, DBF, rx, gx, register, _tbl, _persist, preview,
#   _tile_range, _window_tiles, _lonlat_to_3857, _h3_center, X0, Y0, X1, Y1,
#   TILE_M, SF_AOI, CATALOG, SCHEMA, H3_RESOLUTION=10.
# nb4 sets its own tower/coverage resolution (9) — independent of config's
# H3_RESOLUTION=10 (which is nb3's DEM/DSM gridding resolution).
import inspect

OUT_PREFIX       = "wc_h3"    # coverage table: wc_h3_coverage[_demo]_res10
TOWER_RES        = 9          # H3 resolution for tower lattice
COVERAGE_RES     = 10          # H3 res for coverage output (finer than tower lattice)
TOWER_HEIGHT_M   = 30.0       # tower height above ground DEM, metres
TARGET_HEIGHT_M  = 1.6        # target height above DSM (person at ground), metres

# Viewshed radius. The DEMO uses 1 km so each tower's merged DSM window stays
# within Serverless per-task memory: a 1 km window of 1 m float32 pixels is
# ~37 MB, and the footprint grows with the square of the radius. For a wider
# radius, run on a classic cluster with more executor memory, or downsample
# the DSM further via DSM_VIEWSHED_RES_M below.
MAX_DISTANCE_M   = 1000.0     # viewshed radius, metres

# The viewshed runs on the DSM downsampled to this resolution. 10 m keeps each
# tower's working set small so many viewshed workers run concurrently on
# Serverless, while preserving cell-level line-of-sight: a res-10 coverage
# cell (~65 m) still spans ~6-7 samples per edge.
DSM_VIEWSHED_RES_M = 10.0    # viewshed DSM resolution, metres

DEMO             = True       # True → Golden Gate Park ≈25 towers
INTERACTIVE      = False      # True → deck.gl live render; False → static
FORCE_RELOAD     = False      # True → overwrite wc_h3_coverage; False → reuse

# fan-out: repartition(N, "tower_cellid") for viewshed fan-out.
# Keep low for DEMO to reduce concurrent xrspatial workers and avoid RAM contention.
N_PARTITIONS = 5 if DEMO else 200

# Surface source tables — full SF (nb4 windows by tower AOI; no _demo suffix).
# These are the nb2-produced raster surface tables, not demo-scoped.
DSM_TABLE = f"{CATALOG}.{SCHEMA}.wc_surface_dsm"
DTM_TABLE = f"{CATALOG}.{SCHEMA}.wc_surface_dtm"

# DEMO tile window (Golden Gate Park centroid, ±2 tiles — matches nb3 DEMO_FILTER).
# X0, Y0, TILE_M come from config_nb.
if DEMO:
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)
    DEMO_TX = int((_gx - X0) / TILE_M)
    DEMO_TY = int((_gy - Y0) / TILE_M)
    DEMO_RADIUS_TILES = 2
    print(
        f"DEMO tile window: tx∈[{DEMO_TX-DEMO_RADIUS_TILES},{DEMO_TX+DEMO_RADIUS_TILES}] "
        f"ty∈[{DEMO_TY-DEMO_RADIUS_TILES},{DEMO_TY+DEMO_RADIUS_TILES}]"
    )

## T0 — Preconditions: API verification and dependency check

This section runs live checks on env6 before any implementation cells. Findings are
recorded in `.superpowers/sdd/2026-10-02-wireless-coverage-nb4/t0-preconditions.md`.

In [ ]:
# T0.1 — Confirm input tables exist.
# Raster surface tables: no res/demo suffix (nb2 output, full SF).
# H3 tables from nb3: include _res{H3_RESOLUTION} suffix (config_nb H3_RESOLUTION=10).
# Check full-SF nb3 output (either full or demo variant is acceptable).
for _t in [
    f"{CATALOG}.{SCHEMA}.wc_surface_dsm",
    f"{CATALOG}.{SCHEMA}.wc_surface_dtm",
]:
    assert spark.catalog.tableExists(_t), f"BLOCKER: table missing — run nb2 first: {_t}"
for _name in ["dem", "dsm"]:
    _full = f"{CATALOG}.{SCHEMA}.wc_h3_{_name}_res{H3_RESOLUTION}"
    _demo = f"{CATALOG}.{SCHEMA}.wc_h3_{_name}_demo_res{H3_RESOLUTION}"
    assert (
        spark.catalog.tableExists(_full) or spark.catalog.tableExists(_demo)
    ), f"BLOCKER: nb3 H3 table missing — run nb3 first (expected {_full} or {_demo})"
print("T0.1 — All required input tables exist: OK")

In [ ]:
# T0.2 — h3_centeraswkb on Serverless env6 (Photon/Databricks SQL check)
# Returns: _CENTER_FN = 'product' if h3_centeraswkb works; 'h3_python' as fallback.
try:
    spark.range(1).select(DBF.h3_centeraswkb(F.lit(599686042433355775))).limit(1).collect()
    print("T0.2 — h3_centeraswkb: OK (Photon/Databricks SQL available on this env)")
    _CENTER_FN = "product"     # use DBF.h3_centeraswkb
except Exception as _e:
    print(f"T0.2 — h3_centeraswkb NOT available ({type(_e).__name__}: {_e}); using h3 Python fallback")
    _CENTER_FN = "h3_python"   # use _h3_center from config_nb
print(f"  _CENTER_FN = '{_CENTER_FN}'")

In [ ]:
# T0.3 — rst_viewshed signature: confirm 'crs' and 'materialize' parameters exist
sig_vs = inspect.signature(rx.rst_viewshed)
assert "crs" in sig_vs.parameters, "BLOCKER: rst_viewshed has no 'crs' parameter"
assert "materialize" in sig_vs.parameters, "BLOCKER: rst_viewshed has no 'materialize' parameter"
print(f"T0.3 — rst_viewshed signature: {sig_vs}")
print("  crs + materialize parameters confirmed: OK")

In [ ]:
# T0.5 — rst_merge_agg and rst_sample callable
assert callable(rx.rst_merge_agg), "BLOCKER: rst_merge_agg not callable"
assert callable(rx.rst_sample),    "BLOCKER: rst_sample not callable"
print("T0.5 — rst_merge_agg: OK | rst_sample: OK")

In [ ]:
# T0.6 — gbx_rst_h3_rastertogridmax LATERAL on one DSM tile (functional check)
# ANSI LATERAL form: FROM <table>, LATERAL <udtf>(...) <alias>
# (comma after table is required; matches nb3 verified pattern)
_dsm1 = (
    spark.table(f"{CATALOG}.{SCHEMA}.wc_surface_dsm")
    .limit(1)
    .select(rx.rst_transform("dsm", F.lit(4326)).alias("dsm_4326"))
)
_dsm1.createOrReplaceTempView("_t0_dsm")
_r = spark.sql("""
    SELECT t.cellID, t.measure, t.band
    FROM   _t0_dsm,
           LATERAL gbx_rst_h3_rastertogridmax(dsm_4326, 9) t
    LIMIT 5
""").collect()
assert len(_r) > 0, "BLOCKER: gbx_rst_h3_rastertogridmax returns no rows on DSM tile"
print(f"T0.6 — gbx_rst_h3_rastertogridmax LATERAL on DSM: OK ({len(_r)} sample rows)")
print(f"  Sample: cellID={_r[0]['cellID']}, measure={_r[0]['measure']:.2f}, band={_r[0]['band']}")

In [ ]:
# T0.7 — wc_surface_dsm schema and AOI origin
print("T0.7 — wc_surface_dsm schema:")
spark.table(f"{CATALOG}.{SCHEMA}.wc_surface_dsm").printSchema()

from databricks.labs.gbx.sample.lidar import aoi_lonlat_to_3857 as _aoi_fn
_SF_AOI_CHECK = (-122.55, 37.70, -122.35, 37.85)
_X0c, _Y0c, _X1c, _Y1c = _aoi_fn(*_SF_AOI_CHECK)
print(f"  AOI 3857 origin: X0={_X0c:.0f} Y0={_Y0c:.0f} X1={_X1c:.0f} Y1={_Y1c:.0f}")
print("T0.7 — Schema and AOI origin: OK")

## T1 — Source table loads

Load DSM/DTM from the full-SF raster surface tables (nb2 output). DEMO mode scopes the
tiles to the Golden Gate Park tile window; nb4 does NOT require a `_demo`-suffixed table.

In [ ]:
dsm_full = spark.table(DSM_TABLE)
dtm_full = spark.table(DTM_TABLE)
if DEMO:
    DEMO_FILTER = (
        F.col("tx").between(DEMO_TX - DEMO_RADIUS_TILES, DEMO_TX + DEMO_RADIUS_TILES) &
        F.col("ty").between(DEMO_TY - DEMO_RADIUS_TILES, DEMO_TY + DEMO_RADIUS_TILES)
    )
    dsm_in = dsm_full.where(DEMO_FILTER)
    dtm_in = dtm_full.where(DEMO_FILTER)
else:
    dsm_in, dtm_in = dsm_full, dtm_full
n_dsm = dsm_in.count()
n_dtm = dtm_in.count()
print(f"DSM tiles: {n_dsm} | DTM tiles: {n_dtm}")
assert n_dsm > 0, "No DSM tiles loaded — check DEMO filter or nb2 tables"
assert n_dtm > 0, "No DTM tiles loaded"

In [ ]:
# AOI WKB for the tower lattice (EPSG:4326 bounding box).
# _box and _wkb are imported by config_nb (shapely); pyproj needed here for EPSG:3857→4326.
import pyproj

W, S, E, N = SF_AOI
if DEMO:
    # DEMO AOI: tile window extent in lon/lat (matches DEMO_FILTER)
    _t4326 = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    _tx_min_m = X0 + (DEMO_TX - DEMO_RADIUS_TILES) * TILE_M
    _tx_max_m = X0 + (DEMO_TX + DEMO_RADIUS_TILES + 1) * TILE_M
    _ty_min_m = Y0 + (DEMO_TY - DEMO_RADIUS_TILES) * TILE_M
    _ty_max_m = Y0 + (DEMO_TY + DEMO_RADIUS_TILES + 1) * TILE_M
    W, S = _t4326.transform(_tx_min_m, _ty_min_m)
    E, N = _t4326.transform(_tx_max_m, _ty_max_m)
_aoi_geom = _box(W, S, E, N)
AOI_WKB = _aoi_geom.wkb
print(f"Tower lattice AOI: ({W:.4f}, {S:.4f}, {E:.4f}, {N:.4f})")

## T2 — Tower lattice: H3 cells at TOWER_RES covering the AOI

In [ ]:
# h3_try_coverash3(geom_wkb, res) returns ARRAY<LONG> of H3 cell IDs (non-exception variant).
# Confirmed: h3_rasterize_isobands.ipynb lines 759, 811.
tower_cells = (
    spark.createDataFrame([(1,)], "g int")
    .select(
        F.explode(
            DBF.h3_try_coverash3(F.lit(AOI_WKB), F.lit(TOWER_RES))
        ).alias("tower_cellid")
    )
    .where(F.col("tower_cellid").isNotNull())
)
n_towers = tower_cells.count()
print(f"Tower lattice: {n_towers} H3 cells at res {TOWER_RES} covering the AOI")
# Expected: ~25 (DEMO) / ~1,150 (full SF) at res 9

In [ ]:
# H3 centroid WKB and lon/lat extraction.
# h3_centeraswkb(cellid) → WKB POINT in EPSG:4326.
# If Photon unavailable (T0 flagged _CENTER_FN == 'h3_python'), use _h3_center from config_nb.

if _CENTER_FN == "product":
    towers_wkb = tower_cells.withColumn(
        "center_wkb", DBF.h3_centeraswkb("tower_cellid")
    )
    # Extract lon/lat from the WKB point for subsequent 3857 arithmetic.
    # DBF.st_geomfromwkb + st_x/st_y confirmed in vapor-eyes (cells 62-64).
    towers_latlon = (
        towers_wkb
        .withColumn("_geom", DBF.st_geomfromwkb("center_wkb"))
        .withColumn("lon",   DBF.st_x("_geom"))
        .withColumn("lat",   DBF.st_y("_geom"))
        .drop("_geom")
    )
else:
    # h3 Python UDF fallback: _h3_center from config_nb returns (lat, lon, wkb).
    # config_nb cell-13 defines _h3_center using h3.cell_to_latlng.
    towers_latlon = (
        tower_cells
        .withColumn("_c", _h3_center("tower_cellid"))
        .withColumn("lat", F.col("_c.lat"))
        .withColumn("lon", F.col("_c.lon"))
        .withColumn("center_wkb", F.col("_c.wkb"))
        .drop("_c")
    )

towers_latlon.show(5)

In [ ]:
# lon/lat → EPSG:3857 via _lonlat_to_3857 from config_nb (pure-math Mercator,
# avoids st_transform Photon dependency). config_nb cell-12 defines the UDF.
towers_df = (
    towers_latlon
    .withColumn("_xy",     _lonlat_to_3857("lon", "lat"))
    .withColumn("tower_x", F.col("_xy.x"))   # EPSG:3857 x, metres
    .withColumn("tower_y", F.col("_xy.y"))   # EPSG:3857 y, metres
    .drop("_xy")
)
towers_df.show(5)

In [ ]:
# Sanity: all tower_x/y in SF's expected EPSG:3857 range
_tx_stats = towers_df.selectExpr(
    "min(tower_x)", "max(tower_x)", "min(tower_y)", "max(tower_y)"
).first()
print(
    f"Tower 3857 range: x=[{_tx_stats[0]:.0f}, {_tx_stats[1]:.0f}] "
    f"y=[{_tx_stats[2]:.0f}, {_tx_stats[3]:.0f}]"
)
# SF 3857: x ≈ -13647000 to -13627000, y ≈ 4539000 to 4557000
assert _tx_stats[0] < -13620000, f"tower_x too large ({_tx_stats[0]:.0f}) — CRS conversion likely wrong"
assert _tx_stats[3] < 4560000,   f"tower_y max too large ({_tx_stats[3]:.0f})"
assert _tx_stats[2] > 4535000,   f"tower_y min too small ({_tx_stats[2]:.0f})"
assert towers_df.count() == n_towers, "Tower count changed after 3857 conversion"
print(f"Tower 3857 coordinates — sanity check OK (n={n_towers})")

## T3 — Cross-tile DSM windowing helper (TDD)

For each tower, range-join to DSM tiles whose tx/ty index falls within `MAX_DISTANCE_M`, then
merge into one DSM tile per tower via `rst_merge_agg`. Arithmetic tile-index range join avoids
any ST_ geometry dependency. Helper functions `_tile_range` and `_window_tiles`
are defined in `config_nb` (cells 10–11) and unit-tested at import time.

In [ ]:
# Windowing cross-tile verification: check that at least one tower produces a
# multi-tile window (not a single-tile no-op), proving the range-join works.
_R = MAX_DISTANCE_M
_towers_ranged_check = towers_df.select(
    "tower_cellid",
    F.floor((F.col("tower_x") - X0 - _R) / TILE_M).cast("int").alias("tx_min"),
    F.floor((F.col("tower_x") - X0 + _R) / TILE_M).cast("int").alias("tx_max"),
    F.floor((F.col("tower_y") - Y0 - _R) / TILE_M).cast("int").alias("ty_min"),
    F.floor((F.col("tower_y") - Y0 + _R) / TILE_M).cast("int").alias("ty_max"),
).withColumn(
    "tiles_in_window",
    (F.col("tx_max") - F.col("tx_min") + 1) * (F.col("ty_max") - F.col("ty_min") + 1)
)
_win_stats = _towers_ranged_check.selectExpr(
    "min(tiles_in_window) as min_tiles",
    "max(tiles_in_window) as max_tiles",
    "avg(tiles_in_window) as avg_tiles",
).first()
print(
    f"Window sizes: min={_win_stats['min_tiles']} max={_win_stats['max_tiles']} "
    f"avg={_win_stats['avg_tiles']:.1f} tiles/tower"
)
# At 3 km radius / 1024 m tile, each tower should span ~(6000/1024+1)^2 ≈ ~49 tiles.
assert _win_stats['max_tiles'] > 1, (
    "All towers map to a single tile — range-join is not spanning multiple tiles. "
    "Check TILE_M and MAX_DISTANCE_M values."
)
print("Cross-tile windowing geometry check: PASSED — windows span multiple tiles")

In [ ]:
_t0_win = time.time()
dsm_windows_df = _window_tiles(towers_df, dsm_in, MAX_DISTANCE_M, X0, Y0, TILE_M)
dsm_windows_df = dsm_windows_df.repartition(N_PARTITIONS, "tower_cellid")
# Do NOT .cache() merged raster tiles on Serverless (they OOM).
# .cache() triggers immediate materialization of all merged tiles simultaneously;
# at 3 km radius / 1 m px, each tile is ~36 MB; caching 25+ tiles OOMs workers.
# dsm_windows_df is lazy — the merge runs per-tower on demand in T4 and T5.
# Count windows using the pre-merge range join (avoids triggering rst_merge_agg for count).
_R = MAX_DISTANCE_M
_n_windows_check = (
    towers_df.select(
        "tower_cellid",
        F.floor((F.col("tower_x") - X0 - _R) / TILE_M).cast("int").alias("tx_min"),
        F.floor((F.col("tower_x") - X0 + _R) / TILE_M).cast("int").alias("tx_max"),
        F.floor((F.col("tower_y") - Y0 - _R) / TILE_M).cast("int").alias("ty_min"),
        F.floor((F.col("tower_y") - Y0 + _R) / TILE_M).cast("int").alias("ty_max"),
    )
    .join(dsm_in, (
        (F.col("tx") >= F.col("tx_min")) & (F.col("tx") <= F.col("tx_max")) &
        (F.col("ty") >= F.col("ty_min")) & (F.col("ty") <= F.col("ty_max"))
    ), "inner")
    .select("tower_cellid").distinct()
)
n_windows = _n_windows_check.count()
print(f"DSM windows: {n_windows} towers have DSM coverage in {time.time() - _t0_win:.1f}s")
assert n_windows == n_towers, (
    f"Expected {n_towers} towers with DSM coverage, got {n_windows}. "
    "Missing → some tower centroids fall outside the DSM extent."
)

In [ ]:
# Merged tile size note.
# MAX_DISTANCE_M = 3000 m → estimated merged tile size: ~6000×6000 px = ~36 MB uncompressed.
# At GDAL DEFLATE compression (typical for DSM tiles), expect ~5-15 MB per tile.
# This is within the 64 MB Serverless per-task materialization cap.
# Actual tile size is measured in T5 when rst_viewshed materializes each tile.
# dsm_windows_df is lazy — no merge happens here.
_est_px = int((2 * MAX_DISTANCE_M + TILE_M) ** 2)
_est_mb_uncompressed = _est_px / 1024 / 1024   # uint8 = 1 byte/px
print(
    f"Estimated merged tile footprint: ~{int(2*MAX_DISTANCE_M+TILE_M)}×{int(2*MAX_DISTANCE_M+TILE_M)} px "
    f"= ~{_est_mb_uncompressed:.0f} MB uncompressed. "
    f"Expect ~5-15 MB compressed (DEFLATE)."
)
if _est_mb_uncompressed > 64:
    print(
        f"  WARNING: uncompressed estimate exceeds 64 MB Serverless cap. "
        "Reduce MAX_DISTANCE_M if rst_merge_agg OOMs in T5."
    )
print("dsm_windows_df ready (lazy — merge runs per-tower in T4/T5)")

## T4 — Tower elevation: observer height above the DSM

Each tower is modelled as a point `TOWER_HEIGHT_M` metres above the bare-earth DEM. Because
`rst_viewshed` measures `observer_height` *above the surface raster* (the DSM), the effective
height passed is:

    observer_height = (DEM_elevation + TOWER_HEIGHT_M) − DSM_elevation_at_tower

Where DSM elevation ≈ DEM at open ground sites; the correction matters mainly for towers
that would otherwise sit inside buildings or tree canopies. Values are clamped to 0 (a
tower below the DSM surface is treated as ground level).

**CRS note:** `rst_sample` with `crs=4326` reprojects the observer point (EPSG:4326 WKB
from `h3_centeraswkb`) to the tile CRS (3857) internally — the same reprojection mechanism
for `rst_viewshed`.

In [ ]:
# Join towers to DTM home tile (the tile containing the tower centroid — 1:1 join).
# home_tx / home_ty = the tile whose lower-left corner bounds the tower centroid.
towers_with_tile = towers_df.select(
    "tower_cellid", "center_wkb", "tower_x", "tower_y",
    "lon", "lat",    # carried through for sampling + the render
    F.floor((F.col("tower_x") - X0) / TILE_M).cast("int").alias("home_tx"),
    F.floor((F.col("tower_y") - Y0) / TILE_M).cast("int").alias("home_ty"),
)
dtm_keyed = dtm_in.select(
    F.col("tx").alias("home_tx"),
    F.col("ty").alias("home_ty"),
    F.col("dtm").alias("dtm_tile"),
)
towers_dtm = towers_with_tile.join(dtm_keyed, ["home_tx", "home_ty"], "left")

# rst_sample(tile, geom, crs=4326): returns ARRAY<DOUBLE>; .getItem(0) for band 1.
# crs=4326 → observer point (EPSG:4326 WKB from h3_centeraswkb) reprojected to tile CRS (3857).
# rst_sample returns the raster nodata fill (e.g. -9999) where the bare-earth DTM has
# no ground return (dense buildings) or over water; treat any non-physical sample as
# nodata (NULL) so the tower falls back to a flat-ground height in T4 rather than a
# spurious observer_height of 0.
_DEM_NODATA = -100.0   # SF bare earth is >= ~0 m; -9999 is the DTM nodata fill
towers_sampled = (
    towers_dtm
    .withColumn("_dem_raw", rx.rst_sample("dtm_tile", "center_wkb", F.lit(4326)).getItem(0))
    .withColumn("dem_elev", F.when(F.col("_dem_raw") > _DEM_NODATA, F.col("_dem_raw")))
    .drop("dtm_tile", "_dem_raw")
)

In [ ]:
# Sample DSM elevation at tower centroid from the HOME tile (single-tile, no merge).
# The merged dsm_windows_df (rst_merge_agg) is reserved for T5 rst_viewshed.
# For a point sample at the centroid, the home tile is sufficient and avoids
# triggering the full merge in T4 (the merge OOMs on Serverless when
# chained into a T4 aggregate action).
dsm_keyed = dsm_in.select(
    F.col("tx").alias("home_tx"),
    F.col("ty").alias("home_ty"),
    F.col("dsm").alias("dsm_home_tile"),
)
towers_sampled = (
    towers_sampled
    .join(dsm_keyed, ["home_tx", "home_ty"], "left")
    .withColumn("_dsm_raw", rx.rst_sample("dsm_home_tile", "center_wkb", F.lit(4326)).getItem(0))
    .withColumn("dsm_elev", F.when(F.col("_dsm_raw") > _DEM_NODATA, F.col("_dsm_raw")))
    .drop("dsm_home_tile", "_dsm_raw")
)

In [ ]:
# observer_height = max(0, DEM_elev + TOWER_HEIGHT_M - DSM_elev)
# Clamp to 0: a tower inside a building/canopy is treated as ground level.
towers_with_height = towers_sampled.withColumn(
    "observer_height",
    F.greatest(
        F.lit(0.0),
        F.col("dem_elev") + F.lit(TOWER_HEIGHT_M) - F.col("dsm_elev"),
    ),
)

# Observer-height distribution
_oh_stats = towers_with_height.selectExpr(
    "count(*) as n_towers",
    "sum(case when observer_height = 0.0 then 1 else 0 end) as n_inside_dsm",
    "min(observer_height) as min_oh",
    "max(observer_height) as max_oh",
    "avg(observer_height) as avg_oh",
).first()
print(
    f"Observer height stats: min={_oh_stats['min_oh']:.1f} m  "
    f"max={_oh_stats['max_oh']:.1f} m  avg={_oh_stats['avg_oh']:.1f} m  "
    f"towers inside DSM surface (oh=0): {_oh_stats['n_inside_dsm']}"
)

In [ ]:
# Handle null elevations: towers where DEM or DSM sampling fails (tile boundary edge
# cases) fall back to TOWER_HEIGHT_M above a flat surface.
towers_with_height = towers_with_height.withColumn(
    "observer_height",
    F.when(
        F.col("dem_elev").isNull() | F.col("dsm_elev").isNull(),
        F.lit(TOWER_HEIGHT_M),   # assume flat ground if sampling fails
    ).otherwise(F.col("observer_height")),
)
n_null_elev = towers_with_height.where(
    F.col("dem_elev").isNull() | F.col("dsm_elev").isNull()
).count()
if n_null_elev > 0:
    print(
        f"WARNING: {n_null_elev} towers have null DEM/DSM elevation — "
        f"using TOWER_HEIGHT_M={TOWER_HEIGHT_M} m fallback."
    )

# Final validation
n_final = towers_with_height.count()
n_null_oh = towers_with_height.where(F.col("observer_height").isNull()).count()
assert n_final == n_towers, f"Expected {n_towers} rows, got {n_final}"
assert n_null_oh == 0, f"{n_null_oh} towers still have null observer_height after fallback"
print(
    f"T4 validation PASSED: {n_final} towers with non-null observer_height | "
    f"{n_null_elev} null-elevation fallbacks"
)
towers_with_height.select(
    "tower_cellid", "lon", "lat", "dem_elev", "dsm_elev", "observer_height"
).show(5)

# ── Stage materialisation: T4 output ─────────────────────────────────────
# Persist tower lattice + centroids + observer elevations so re-runs skip T2–T4.
# _persist gates on FORCE_RELOAD; with FORCE_RELOAD=False a 2nd run reads the table.
# Excludes binary WKB column (home_tx/ty are internal; carry through lon/lat + elev).
towers_stage = _persist(
    towers_with_height.select(
        "tower_cellid", "center_wkb", "lon", "lat", "tower_x", "tower_y",
        "dem_elev", "dsm_elev", "observer_height"
    ),
    "towers",
    res=TOWER_RES,
)
print(f"Towers stage table: {towers_stage.count()} rows → {_tbl(OUT_PREFIX, 'towers', TOWER_RES)}")

## T5–T6 — Viewshed per tower → per-cell visibility

**Stage gate:** if `wc_h3_visible_pairs[_demo]_res9` already exists and `FORCE_RELOAD=False`,
the entire viewshed + H3-re-tessellation stage is skipped and results are read from the
persisted table. Re-runs are therefore snappy (sub-minute for DEMO).

If FORCE_RELOAD=True, the stage is recomputed:

1. **T5 `rst_viewshed`** — for each tower, resample DSM to `DSM_VIEWSHED_RES_M` (10 m) to
   bound working memory within Serverless constraints, then compute a binary LOS mask
   (255=visible, 0=blocked) via `xrspatial.viewshed`.
2. **T6 `gbx_rst_h3_rastertogridmax`** — reproject viewshed to EPSG:4326 and tessellate
   visible pixels to H3 cells at `TOWER_RES` via LATERAL UDTF.

In [ ]:
_visible_pairs_tbl = _tbl(OUT_PREFIX, "visible_pairs", res=COVERAGE_RES)
_skip_viewshed = (not FORCE_RELOAD) and spark.catalog.tableExists(_visible_pairs_tbl)

if _skip_viewshed:
    print(f"Stage gate: reading from {_visible_pairs_tbl} (FORCE_RELOAD=False, table exists)")
    tower_visible_pairs = spark.table(_visible_pairs_tbl)
    n_pairs = tower_visible_pairs.count()
    print(f"  (tower, visible_cell) pairs: {n_pairs}")
else:
    # ── T5: rst_viewshed per tower ──────────────────────────────────────────
    # Join towers_stage (T4 persisted) with dsm_windows_df.
    towers_for_viewshed = towers_stage.join(
        dsm_windows_df.select("tower_cellid", "dsm_window"),
        "tower_cellid",
        "inner",
    )

    # Downsample the merged DSM to DSM_VIEWSHED_RES_M before the viewshed so each
    # tower's working set stays small enough for concurrent Serverless workers.

    _t5_start = time.time()
    viewshed_df = (
        towers_for_viewshed
        .repartition(N_PARTITIONS, "tower_cellid")
        .withColumn(
            "dsm_coarse",
            rx.rst_resample_to_res(
                "dsm_window",
                F.lit(DSM_VIEWSHED_RES_M),
                F.lit(DSM_VIEWSHED_RES_M),
                F.lit("bilinear"),      # bilinear for continuous elevation data
                materialize=True,       # bytes in struct — prevent virtual tile across shuffle
            ),
        )
        .withColumn(
            "viewshed_tile",
            rx.rst_viewshed(
                "dsm_coarse",           # downsampled DSM for memory-safe viewshed
                "center_wkb",           # observer WKB in EPSG:4326 (h3_centeraswkb output)
                "observer_height",      # height above DSM surface (from T4 stage table)
                F.lit(TARGET_HEIGHT_M), # target height (person at ground), metres
                F.lit(MAX_DISTANCE_M),  # viewshed radius cap
                F.lit(4326),            # CRS of observer_geom (WKB is EPSG:4326)
                materialize=True,       # bytes in struct — required for subsequent rst_transform
            ),
        )
        .drop("dsm_coarse")
    )

    n_viewsheds = viewshed_df.where(F.col("viewshed_tile").isNotNull()).count()
    n_null_vs   = viewshed_df.where(F.col("viewshed_tile").isNull()).count()
    print(
        f"T5 viewsheds: {n_viewsheds} non-null, {n_null_vs} null "
        f"in {time.time() - _t5_start:.1f}s"
    )
    assert n_viewsheds > 0, (
        "BLOCKER: all viewshed tiles are null — check CRS, observer_height, and DSM window extent"
    )

    # ── T5 spot-check ──────────────────────────────────────────────────────
    _vs_sample_row = viewshed_df.select("tower_cellid", "viewshed_tile").limit(1).first()
    if _vs_sample_row and _vs_sample_row["viewshed_tile"]:
        _tile_keys = list(_vs_sample_row["viewshed_tile"].asDict().keys()) \
                     if hasattr(_vs_sample_row["viewshed_tile"], "asDict") else "(row)"
        print(f"T5 sample viewshed — tower_cellid: {_vs_sample_row['tower_cellid']}, fields: {_tile_keys}")
    else:
        raise AssertionError("T5 sample viewshed is null — observer likely outside DSM extent")

    # ── T6: viewshed → H3 cells ────────────────────────────────────────────
    # Step 1: reproject viewshed from EPSG:3857 → EPSG:4326 (required by H3 LATERAL UDTF).
    viewshed_4326_df = (
        viewshed_df
        .repartition(N_PARTITIONS, "tower_cellid")
        .withColumn(
            "viewshed_4326",
            rx.rst_transform("viewshed_tile", F.lit(4326)),
        )
        .select("tower_cellid", "viewshed_4326")
    )
    viewshed_4326_df.createOrReplaceTempView("viewshed_4326_tiles")

    # Step 2: LATERAL gbx_rst_h3_rastertogridmax — visible H3 cells per tower.
    # measure = max pixel value in the H3 cell: 255 = visible, 0 = blocked.
    # WHERE measure > 127 → cell is visible if any pixel in it is visible.
    _t6_start = time.time()
    tower_visible_raw = spark.sql(f"""
        SELECT tower_cellid, t.cellID AS visible_cellid
        FROM   viewshed_4326_tiles,
               LATERAL gbx_rst_h3_rastertogridmax(viewshed_4326, {COVERAGE_RES}) t
        WHERE  t.band = 1 AND t.measure > 127
    """)
    tower_visible_pairs_computed = tower_visible_raw.distinct()

    n_pairs = tower_visible_pairs_computed.count()
    _t6_elapsed = time.time() - _t6_start
    print(f"T6 (tower, visible_cell) pairs: {n_pairs} in {_t6_elapsed:.1f}s")
    avg_vis = n_pairs / n_towers if n_towers > 0 else 0
    print(f"  Average visible cells per tower: {avg_vis:.0f}")
    assert n_pairs > 0, "No visible cell pairs — check viewshed CRS or tile content"

    # ── Stage materialisation: T6 output (critical) ─────────────────────
    tower_visible_pairs = _persist(tower_visible_pairs_computed, "visible_pairs", res=COVERAGE_RES)
    print(f"Visible pairs staged: {tower_visible_pairs.count()} rows → {_visible_pairs_tbl}")

## T7 — Aggregate to `wc_h3_coverage`

Group (tower, visible_cell) pairs by cell, count distinct towers per cell → `tower_los_count`.
Persist as `wc_h3_coverage[_demo]_res10` via `_persist` from `config_nb`.

In [ ]:
coverage_df = (
    tower_visible_pairs
    .groupBy(F.col("visible_cellid").alias("cellid"))
    .agg(F.countDistinct("tower_cellid").alias("tower_los_count"))
    .withColumn("res",     F.lit(COVERAGE_RES))
    .withColumn("covered", F.lit(True))   # all cells in the table have ≥1 sightline
    .select("cellid", "res", "tower_los_count", "covered")
)
n_covered = coverage_df.count()
print(f"Coverage: {n_covered} H3 cells covered by ≥1 tower sightline")
print(f"Tower lattice size: {n_towers} cells")
coverage_df.selectExpr(
    "min(tower_los_count)", "max(tower_los_count)", "avg(tower_los_count)"
).show()

In [ ]:
# _persist from config_nb: uses OUT_PREFIX ('wc_h3'), DEMO flag, and res suffix.
# With DEMO=True, COVERAGE_RES=10: writes geospatial_docs.geobrix.wc_h3_coverage_demo_res10.
# With DEMO=False, COVERAGE_RES=10: writes geospatial_docs.geobrix.wc_h3_coverage_res10.
coverage_h3 = _persist(coverage_df, "coverage", res=COVERAGE_RES)
n_persisted = coverage_h3.count()
print(f"wc_h3_coverage persisted: {n_persisted} covered cells")

expected_cols = {"cellid", "res", "tower_los_count", "covered"}
actual_cols   = set(coverage_h3.columns)
assert expected_cols == actual_cols, f"Schema mismatch: expected {expected_cols}, got {actual_cols}"
print("wc_h3_coverage schema OK")

print(
    f"\nSummary (DEMO={DEMO}):\n"
    f"  Tower resolution: H3 res {TOWER_RES}\n"
    f"  Coverage res:     H3 res {COVERAGE_RES}\n"
    f"  Tower height:     {TOWER_HEIGHT_M} m above DEM\n"
    f"  Viewshed radius:  {MAX_DISTANCE_M} m\n"
    f"  Towers:           {n_towers}\n"
    f"  Covered cells:    {n_covered}\n"
    f"  Max LOS count:    {coverage_h3.selectExpr('max(tower_los_count)').first()[0]}\n"
)

## T8 — Coverage maps

Three views of the result. **The output tables are the deliverable, not just these plots:**
`wc_h3_coverage[_demo]_res10` — plus the staged `wc_h3_towers` and `wc_h3_visible_pairs` —
are H3 cell-id tables ready to hand off to CARTO or any H3-aware renderer.

1. **Tower sites** — where the towers are and how high they sit.
2. **Coverage heatmap** — line-of-sight tower count per cell.
3. **Coverage + towers** — the two together.

Static `plot_static` renders (GitHub-safe); set `INTERACTIVE=True` for deck.gl live views.


### 1. Tower sites and elevation

Each tower is the centroid of an H3 res-9 cell. This plots the actual tower **points**,
coloured by **tower-top elevation** — the bare-earth DEM at the site plus the tower height.
Towers over DTM gaps (no bare-earth return — dense building cores, water) are left uncoloured.


In [ ]:
# Tower sites — the actual antenna points (H3 cell centroids), coloured by
# observer elevation: bare-earth DEM at the site plus the tower height.
_towers_pts = towers_stage.withColumn("tower_top_z", F.col("dem_elev") + F.lit(TOWER_HEIGHT_M))
_t_pts_gdf = _points_as_gdf(_towers_pts, "lon", "lat", value_cols=("tower_top_z",))
_t_pts_elev = _t_pts_gdf.dropna(subset=["tower_top_z"])   # colour only where bare earth resolved
_towers_ax = plot_static(
    _t_pts_elev,
    column="tower_top_z",
    cmap="plasma",
    markersize=45,
    title=(
        f"Tower sites — {'DEMO: Golden Gate Park' if DEMO else 'Full SF'} "
        f"(res {TOWER_RES}; colour = tower-top elevation = DEM + {TOWER_HEIGHT_M:.0f} m, where bare earth resolved)"
    ),
)
_towers_png = f"/Volumes/{CATALOG}/geobrix/sample-data/wc_nb4_towers_demo.png"
try:
    _fig = _towers_ax.figure if hasattr(_towers_ax, "figure") else None
    if _fig is not None:
        _fig.savefig(_towers_png, dpi=150, bbox_inches="tight")
        print(f"Tower-sites render saved: {_towers_png}")
    else:
        import matplotlib.pyplot as plt
        plt.savefig(_towers_png, dpi=150, bbox_inches="tight")
        print(f"Tower-sites render saved (plt.gcf): {_towers_png}")
except Exception as _e:
    print(f"WARNING: could not save render ({type(_e).__name__}: {_e})")

### 2. Coverage — tower line-of-sight count

For each res-10 cell, how many towers have a clear line of sight to it (the per-tower viewsheds
from T5–T6, aggregated in T7). Darker = more towers see the cell = more redundant coverage.
This `wc_h3_coverage` table is the primary CARTO handoff.


In [ ]:
# Coverage heatmap — colored by tower_los_count.
_c_gdf = cells_as_gdf(
    coverage_h3.select("cellid", "tower_los_count"),
    "cellid",
    extra_cols=("tower_los_count",),
)
_coverage_ax = plot_static(
    vector_layer(
        _c_gdf,
        geom_col="geometry",
        column="tower_los_count",
        cmap="YlOrRd",
        opacity=0.7,
        label=f"Tower LOS count (res {COVERAGE_RES})",
    ),
    title=(
        f"H3 coverage — tower line-of-sight count "
        f"({'DEMO' if DEMO else 'Full SF'}, {MAX_DISTANCE_M:.0f} m radius)"
    ),
)
# Save the coverage render immediately so plt.gcf() doesn't get cleared by subsequent calls.
_render_path = f"/Volumes/{CATALOG}/geobrix/sample-data/wc_nb4_coverage_demo.png"
try:
    _fig = _coverage_ax.figure if hasattr(_coverage_ax, "figure") else None
    if _fig is not None:
        _fig.savefig(_render_path, dpi=150, bbox_inches="tight")
        print(f"Coverage render saved: {_render_path}")
    else:
        import matplotlib.pyplot as plt
        plt.savefig(_render_path, dpi=150, bbox_inches="tight")
        print(f"Coverage render saved (plt.gcf): {_render_path}")
except Exception as _e:
    print(f"WARNING: Could not save render ({type(_e).__name__}: {_e})")

### 3. Coverage + tower sites

The coverage heatmap with the tower points overlaid — how tower placement drives the
line-of-sight coverage pattern.


In [ ]:
# Combined overlay: H3 coverage heatmap + the actual tower sites (DEMO only — keeps render fast).
if DEMO:
    _combined_ax = plot_static(
        [
            vector_layer(
                _c_gdf, geom_col="geometry", column="tower_los_count",
                cmap="YlOrRd", opacity=0.6, label="Tower LOS count",
            ),
            vector_layer(
                _t_pts_gdf, geom_col="geometry",
                color="#111111",   # tower sites as dark markers over the coverage
                opacity=0.9, label="Tower sites",
            ),
        ],
        title="DEMO: H3 coverage + tower sites (Golden Gate Park)",
    )
    _combined_png = f"/Volumes/{CATALOG}/geobrix/sample-data/wc_nb4_combined_demo.png"
    try:
        _combined_ax.figure.savefig(_combined_png, dpi=150, bbox_inches="tight")
        print(f"Combined render saved: {_combined_png}")
    except Exception as _e:
        print(f"WARNING: could not save combined render ({type(_e).__name__}: {_e})")